## Installation

!uv add llama-index-{component-type}-{framework-name}

In [4]:
!uv add llama-index-llms-huggingface-api llama-index-embeddings-huggingface

Resolved 363 packages in 1.22s                                       
Installed 3 packages in 13mss-huggingface==0.8.0            
 + llama-index-embeddings-huggingface==0.8.0
 + llama-index-llms-huggingface-api==0.8.0
 + sentence-transformers==6.1.0


In [ ]:
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
import os
from dotenv import load_dotenv

load_dotenv()

hf_token = os.getenv("HF_TOKEN")

llm = HuggingFaceInferenceAPI(
    model_name="Qwen/Qwen2.5-Coder-32B-Instruct",
    temperature=0.7,
    max_tokens=100,
    token=hf_token,
    provider="auto"
)

response = llm.complete("Hello, how are you?")
print(response)


In [5]:
!uv add llama-index-llms-openai

Resolved 363 packages in 19ms
Audited 336 packages in 89ms


In [6]:
from llama_index.llms.openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")

llm = OpenAI(
    model="gpt-4o-mini",
    temperature=0.7,
    max_tokens=100,
    api_key=openai_api_key,
)

response = llm.complete("Hello, how are you?")
print(response)


Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?


## Loading and embedding

In [ ]:
- LlamaHub
- Llamaparse

In [ ]:
from llama_index.core import SimpleDirectoryReader

reader = SimpleDirectoryReader(input_dir="path/to/directory")
documents = reader.load_data()

In [ ]:
IngestionPipeline -> SentenceSplitter -> HuggingFaceEmbedding

In [7]:
from llama_index.core import Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline

# create the pipeline with transformations
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_overlap=0),
        HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5"),
    ]
)

nodes = await pipeline.arun(documents=[Document.example()])

/Users/nehe/projects/ai_agents/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█| 199/199 [00:00<00:00, 7


In [ ]:
nodes


## Storing and indexing documents

In [10]:
!uv add llama-index-vector-stores-chroma

Resolved 381 packages in 7.03s                                       
Prepared 10 packages in 5.80s                                            
Installed 18 packages in 88ms                               
 + bcrypt==5.0.0
 + build==1.6.1
 + chromadb==1.5.9
 + durationpy==0.11
 + flatbuffers==25.12.19
 + httptools==0.9.0
 + importlib-resources==7.1.0
 + kubernetes==36.0.3
 + llama-index-vector-stores-chroma==0.6.0
 + mmh3==5.3.1
 + oauthlib==4.0.0
 + onnxruntime==1.31.0
 + overrides==7.7.0
 + pybase64==1.5.1
 + pypika==0.51.1
 + pyproject-hooks==1.3.3
 + requests-oauthlib==2.0.0
 + uvloop==0.23.0


In [13]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex

db = chromadb.PersistentClient(path="./abv_chroma_db")
chroma_collection = db.get_or_create_collection("flights")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=25, chunk_overlap=0),
        embed_model,
    ],
    vector_store=vector_store,
)

index = VectorStoreIndex.from_vector_store(vector_store, embed_model=embed_model)


Loading weights: 100%|█| 199/199 [00:00<00:00, 7


## Querying a VectorIndex with prompts and LLMs

- as_chat_engine
- as_query_engine
- as_retreiver 

In [23]:
from llama_index.llms.openai import OpenAI

openai_api_key = os.getenv("OPENAI_API_KEY")

llm = OpenAI(
    model="gpt-4o-mini",
    temperature=0.7,
    max_tokens=100,
    api_key=openai_api_key,
)

query_engine = index.as_query_engine(
    llm=llm,
    response_mode="tree_summarize"
)
response = query_engine.query("What is the meaning of coding?")
response

# from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI

# llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")
# query_engine = index.as_query_engine(
#     llm=llm,
# )
# query_engine.query("What is the meaning of coding?")


Response(response='Empty Response', source_nodes=[], metadata=None)

## Response Processing

In [ ]:
- refine
- compact
- tree_summarize

## Evaluation

- CorrectnessEvaluator
- FaithfulnessEvaluator
- AnswerRelevancyEvaluator

In [ ]:
from llama_index.core.evaluation import FaithfulnessEvaluator

evaluator = FaithfulnessEvaluator(llm=llm)
response = query_engine.query(
    "What olympics took place in Beijing where bolt won?"
)
eval_result = evaluator.evaluate_response(response=response)
eval_result.passing